# KAN-24 - Hipoteses Estatisticas Antes dos Testes

**Projeto:** Spaceship Titanic - Analise Estatistica  
**Responsavel:** Daniela  
**Revisora dedicada:** Renatta  
**Base analitica:** `data/processed/spaceship_titanic_clean.csv`

## Objetivo desta etapa

Antes de executar qualquer teste inferencial, registramos as perguntas, as hipoteses e a regra de decisao. Esta ordem e importante: se uma hipotese fosse escolhida depois de observar os resultados, poderiamos favorecer explicacoes que parecem significativas apenas por acaso. Essa pratica e conhecida como *p-hacking*.

Esta entrega **nao calcula testes, valores-p, intervalos de confianca, tamanhos de efeito ou conclusoes**. Esses resultados pertencem as tarefas KAN-25, KAN-26 e KAN-27.

## 1. Preparacao reproduzivel

Usaremos apenas a base de treinamento tratada no KAN-12. Ela contem o desfecho observado `Transported` e as variaveis derivadas, incluindo `TotalSpend`. A base `test.csv` do Kaggle nao possui o desfecho e, por isso, nao serve para estes testes de hipoteses.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Permite executar o notebook a partir da raiz do projeto ou da pasta notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spaceship_titanic_clean.csv'
alpha = 0.05

df = pd.read_csv(DATA_PATH)

print(f'Base carregada: {DATA_PATH}')
print(f'Dimensao: {df.shape[0]:,} passageiros x {df.shape[1]} colunas')
print(f'Nivel de significancia pre-especificado: alpha = {alpha:.2f}')

Base carregada: C:\dev\PosGraduacao\Estatistica\Trabalho Final\spaceship-titanic-estatistica-posdl\data\processed\spaceship_titanic_clean.csv
Dimensao: 8,693 passageiros x 22 colunas
Nivel de significancia pre-especificado: alpha = 0.05


## 2. Contrato dos dados para a inferencia

Antes de formular testes, confirmamos que as variaveis previstas existem, nao possuem ausencias e apresentam as categorias esperadas. Esta verificacao nao e um teste de hipotese: e uma protecao contra analisar a base errada ou uma versao incompleta dela.

In [2]:
REQUIRED_COLUMNS = {'HomePlanet', 'CryoSleep', 'TotalSpend', 'Transported'}
EXPECTED_PLANETS = {'Earth', 'Europa', 'Mars'}

missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f'Colunas obrigatorias ausentes: {sorted(missing_columns)}')

null_counts = df[list(REQUIRED_COLUMNS)].isna().sum()
if (null_counts > 0).any():
    raise ValueError(f'Valores ausentes nas variaveis analiticas: {null_counts[null_counts > 0].to_dict()}')

transported_categories = set(df['Transported'].unique())
if len(transported_categories) != 2:
    raise ValueError(f'Transported deve ter duas categorias; encontrado: {transported_categories}')

observed_planets = set(df['HomePlanet'].unique())
if observed_planets != EXPECTED_PLANETS:
    raise ValueError(
        f'Categorias inesperadas em HomePlanet. Esperado: {EXPECTED_PLANETS}; observado: {observed_planets}'
    )

print('Contrato validado: as quatro variaveis estao completas e disponiveis.')
print('\nContagem por HomePlanet:')
display(df['HomePlanet'].value_counts().rename_axis('HomePlanet').to_frame('n'))
print('\nContagem por CryoSleep:')
display(df['CryoSleep'].value_counts().rename_axis('CryoSleep').to_frame('n'))
print('\nContagem por Transported:')
display(df['Transported'].value_counts().rename_axis('Transported').to_frame('n'))

Contrato validado: as quatro variaveis estao completas e disponiveis.

Contagem por HomePlanet:


,n
HomePlanet,
Earth,4715
Europa,2174
Mars,1804



Contagem por CryoSleep:


,n
CryoSleep,
False,5558
True,3135



Contagem por Transported:


,n
Transported,
True,4378
False,4315


## 3. Conceitos essenciais

Uma **hipotese nula** ($H_0$) representa a ausencia de associacao ou diferenca. A **hipotese alternativa** ($H_1$) representa uma associacao ou diferenca que desejamos investigar.

O nivel de significancia foi fixado em $\alpha = 0{,}05$. Isso limita a 5% o risco planejado de rejeitar $H_0$ quando ela e verdadeira, o chamado **erro do tipo I**. Depois que os testes forem executados, a regra sera:

- se $p < 0{,}05$, rejeitar $H_0$;
- se $p \geq 0{,}05$, nao rejeitar $H_0$.

O valor-p nao e a probabilidade de $H_0$ ser verdadeira. Ele mede quao compativeis os dados observados seriam com um modelo em que $H_0$ fosse verdadeira. Por isso, **nao rejeitar** $H_0$ nao prova igualdade entre os grupos: significa apenas que os dados nao forneceram evidencia suficiente contra ela sob este criterio.

## 4. Pergunta 1 - Planeta de origem e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia conforme o planeta de origem?

- **Variaveis:** `HomePlanet` (Earth, Europa, Mars) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `HomePlanet` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos planetas de origem.
- **$H_1$:** existe associacao entre `HomePlanet` e `Transported`; pelo menos uma distribuicao/proporcao difere.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

O qui-quadrado e adequado porque ambas as variaveis sao categoricas. Ele compara as frequencias observadas em uma tabela de contingencia com as frequencias que seriam esperadas se nao houvesse associacao.

## 5. Pergunta 2 - Criossomo e teletransporte

**Pergunta de negocio:** a proporcao de passageiros teletransportados varia entre quem estava ou nao em criossomo?

- **Variaveis:** `CryoSleep` (True, False) e `Transported` (True, False).
- **Unidade de analise:** um passageiro.
- **$H_0$:** `CryoSleep` e `Transported` sao independentes; a distribuicao de `Transported` e a mesma nos dois estados de criossomo.
- **$H_1$:** existe associacao entre `CryoSleep` e `Transported`; as distribuicoes/proporcoes diferem.
- **Teste previsto:** qui-quadrado de independencia, na KAN-25.

Tambem temos duas variaveis categoricas. O teste respondera se a diferenca observada nas frequencias e grande demais para ser atribuida apenas a variacao amostral sob a hipotese de independencia.

## 6. Pergunta 3 - Gasto total e teletransporte

**Pergunta de negocio:** passageiros teletransportados e nao teletransportados apresentam gasto total a bordo diferente?

- **Variaveis:** `TotalSpend` (quantitativa) e `Transported` (dois grupos).
- **Unidade de analise:** um passageiro.
- **$H_0$:** a distribuicao de `TotalSpend` e igual nos grupos `Transported = True` e `Transported = False`; em particular, nao ha diferenca sistematica de localizacao central entre os grupos.
- **$H_1$:** a distribuicao de `TotalSpend` difere entre os dois grupos; existe diferenca sistematica de localizacao central e/ou de distribuicao.
- **Testes previstos:** teste t de Welch para diferenca de medias e Mann-Whitney U como comparacao nao parametrica, na KAN-26.

`TotalSpend` possui muitos zeros e assimetria a direita. Por isso, a KAN-26 utilizara dois metodos complementares: Welch nao exige variancias iguais, e Mann-Whitney trabalha com a ordenacao dos valores. Quando as formas das distribuicoes forem comparaveis, o Mann-Whitney tambem pode ser interpretado como evidencia sobre diferenca de localizacao/mediana; se nao forem, sua interpretacao principal e sobre diferenca de distribucoes.

## 7. Pergunta 4 - Gasto total e planeta de origem

**Pergunta de negocio:** o gasto total a bordo e o mesmo para passageiros de Earth, Europa e Mars?

- **Variaveis:** `TotalSpend` (quantitativa) e `HomePlanet` (tres grupos).
- **Unidade de analise:** um passageiro.
- **$H_0$:** a distribuicao de `TotalSpend` e igual para Earth, Europa e Mars; nao ha diferenca sistematica de localizacao central entre os tres planetas.
- **$H_1$:** pelo menos um planeta apresenta distribuicao de `TotalSpend` diferente; ha diferenca sistematica entre pelo menos dois grupos.
- **Testes previstos:** ANOVA de Welch para diferenca de medias e Kruskal-Wallis como comparacao nao parametrica, na KAN-27.

A ANOVA responde a pergunta global sobre medias dos tres grupos sem assumir variancias iguais. Kruskal-Wallis oferece uma verificacao nao parametrica baseada em postos. Se a hipotese nula global for rejeitada, a KAN-27 executara comparacoes pos-hoc para identificar quais pares de planetas diferem.

## 8. Limite desta entrega e proximos passos

A KAN-24 estabelece o compromisso metodologico: perguntas, hipoteses e regra de decisao foram definidos antes de consultar resultados inferenciais.

| Proxima tarefa | Entrega esperada |
| --- | --- |
| KAN-25 | Tabelas de contingencia e testes qui-quadrado para as perguntas 1 e 2 |
| KAN-26 | Levene, Welch, Mann-Whitney, intervalos de confianca e tamanhos de efeito para a pergunta 3 |
| KAN-27 | Levene, ANOVA de Welch, Kruskal-Wallis, pos-hoc e tamanhos de efeito para a pergunta 4 |
| KAN-28 | Tabela-resumo consolidada de todos os testes |

Portanto, este notebook para antes de calcular qualquer estatistica inferencial. Essa separacao mantem a analise transparente e protege a interpretacao dos resultados posteriores.